# Music Project
### Jon Louis | Betoya Bundu | Wonni Sumani

## Core Learning Outcomes
##### - Prompt Engineering Mastery: Designing multi-faceted prompts to elicit specific analytical outputs.
##### - API Resilience & Quality Assurance: Implementing industrial-strength retry mechanisms with exponential backoff to handle external API failures gracefully.
##### - Structured Data Extraction: Forcing the LLM to return data in JSON format and writing Python code to parse and integrate that data reliably.
##### - Data Curation: Creating a single, central Python dictionary that is progressively enriched by multi-stage AI analysis

In [ ]:
%pip install -qU 'google-genai>=1.0.0'

# Stage 1.1 - API Handling
##### Secure API key handling; Model initialization.

In [ ]:
#stage 1

# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the Gemini API
gemini_model = genai.GenerativeModel('gemini-flash-latest')

# stage 1.2-3 - Source Data I/O & Initialization
##### Robust File I/O using a try...except block to catch and handle FileNotFoundError.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import csv

song = {}

try:
  with open('/content/drive/MyDrive/Business Applications/Loyalty (1).txt', 'r') as f:
    Loyalty = f.read().split()
except FileNotFoundError as e:
  print(f"The file {e} was not found.")
else:
  print("File Found")

song['song_title'] = Loyalty[0]
song['artist'] = Loyalty[1]
song['lyrics'] = ' '.join(Loyalty[2:])

print(song)


File Found
{'song_title': '\ufeffLOYALTY.', 'artist': 'Kendrick', 'lyrics': 'Lamar I said I\'m geeked and I’m fired up (Fired, fired) All I want tonight is just get high (High, high, high) Girl, you look so good, it\'s to die for (Die for, fire) Ooh, that pussy good, it\'s to die for (I’m on fire) It\'s a secret society All we ask is trust (All we ask is trust) All we got is us Loyalty, loyalty, loyalty Loyalty, loyalty, loyalty Kung Fu Kenny now My résumé is real enough for two millennium A better way to make a wave—stop defendin\' them I meditate and moderate all of my wins again I\'m hangin\' on the fence again I\'m always on your mind I put my lyric and my lifeline on the line And ain\'t no limit when I might shine, might grind You\'re rollin’ with it at the right time, right now (Only for the dollar sign) Bad girl RiRi now Swerve, swerve, swerve, swerve, leave it now On your pulse like it’s EDM Gas in the bitch like it\'s premium Haul ass on a bitch all in the fast lane Been a bad

#stage 2 - Extract Factual Data - Implement API Resilience - Data Integration
##### Request a strict JSON Schema for the response / Handle both general exceptions and specific json.JSONDecodeError to ensure data parsing reliability / Parsing and integration of external data

In [ ]:
#stage 2

import json
import time # Import time module for delays


attempt = 0
max_attempts = 3
delay = 1 # Initial delay in seconds

prompt = f"Find the song writers, Album title and release date for the song dictionary '{song}', in a strict JSON schema"

while attempt < max_attempts:
  try:
    response = gemini_model.generate_content(prompt)
    print(response.text)

    # Extract JSON string, handling cases where it's wrapped in markdown
    json_string = response.text.strip()
    if json_string.startswith("```json") and json_string.endswith("```"):
        json_string = json_string[7:-3].strip()

    # Parse the JSON string into a dictionary and update the song dictionary
    song_data = json.loads(json_string)
    song.update(song_data)
    break # Break loop on success
  except json.JSONDecodeError as e:
    attempt += 1
    print(f"JSON decoding error (attempt {attempt}/{max_attempts}): {e}")
    if attempt < max_attempts:
      print(f"Retrying in {delay} seconds...")
      time.sleep(delay)
      delay *= 2 # Exponential backoff
  except Exception as e:
    attempt += 1
    print(f"Error (attempt {attempt}/{max_attempts}): {e}")
    if attempt < max_attempts:
      print(f"Retrying in {delay} seconds...")
      time.sleep(delay)
      delay *= 2 # Exponential backoff

if not song and attempt == max_attempts:
    print(f"Failed to retrieve song data after {max_attempts} attempts.")
else:
    print("Song data successfully retrieved.")
    print(song)

ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2226.63ms


```json
{
  "song_title": "LOYALTY.",
  "album_title": "DAMN.",
  "release_date": "April 14, 2017",
  "songwriters": [
    "Kendrick Duckworth",
    "Dijon McFarlane",
    "Mark Spears",
    "Tyler Williams"
  ]
}
```
Song data successfully retrieved.
{'song_title': 'LOYALTY.', 'artist': 'Kendrick', 'lyrics': 'Lamar I said I\'m geeked and I’m fired up (Fired, fired) All I want tonight is just get high (High, high, high) Girl, you look so good, it\'s to die for (Die for, fire) Ooh, that pussy good, it\'s to die for (I’m on fire) It\'s a secret society All we ask is trust (All we ask is trust) All we got is us Loyalty, loyalty, loyalty Loyalty, loyalty, loyalty Kung Fu Kenny now My résumé is real enough for two millennium A better way to make a wave—stop defendin\' them I meditate and moderate all of my wins again I\'m hangin\' on the fence again I\'m always on your mind I put my lyric and my lifeline on the line And ain\'t no limit when I might shine, might grind You\'re rollin’ with it

#stage 3 - Qualitative Analysis - Lexical Feature Engineering
##### The output must be a single, valid JSON object containing all six analytical fields / Requires importing and contextually using NLTK stopwords within the prompt's instructions

In [ ]:
#stage 3

import json
import time # Import time module for delays

attempt = 0
max_attempts = 3
delay = 1 # Initial delay in seconds

song_data_json = json.dumps(song)
prompt2 = f"Using the following song data: {song_data_json}, determine the genre(s), if the song is explicit, create a summary, sentiments and topics/themes based on this song data in a valid JSON object containing all six analytical fields."

while attempt < max_attempts:
    try:
        response2 = gemini_model.generate_content(prompt2)

        # Check if response has text content before accessing
        if not response2.text:
            if response2.prompt_feedback and response2.prompt_feedback.safety_ratings:
                safety_reasons = [f"{s.category}: {s.probability}" for s in response2.prompt_feedback.safety_ratings]
                raise ValueError(f"Model response blocked due to safety concerns: {'; '.join(safety_reasons)}")
            else:
                raise ValueError("Model response was empty or blocked for an unknown reason.")

        print(response2.text)

        # Extract JSON string, handling cases where it's wrapped in markdown
        json_string_response2 = response2.text.strip()
        if json_string_response2.startswith("```json") and json_string_response2.endswith("```"):
            json_string_response2 = json_string_response2[7:-3].strip()

        song_analysis = json.loads(json_string_response2)
        print("\nSong analysis successfully retrieved.")
        print(song_analysis)
        break # Break loop on success
    except (json.JSONDecodeError, ValueError) as e:
        attempt += 1
        print(f"Error (attempt {attempt}/{max_attempts}): {e}")
        if attempt < max_attempts:
            print(f"Retrying in {delay} seconds...")
            time.sleep(delay)
            delay *= 2 # Exponential backoff
    except Exception as e:
        attempt += 1
        print(f"An unexpected error occurred (attempt {attempt}/{max_attempts}): {e}")
        if attempt < max_attempts:
            print(f"Retrying in {delay} seconds...")
            time.sleep(delay)
            delay *= 2 # Exponential backoff

if 'song_analysis' not in locals() and attempt == max_attempts:
    print(f"Failed to retrieve song analysis after {max_attempts} attempts.")


prompt3 = f"Using the following song lyrics: {song['lyrics']}, identify the 10 most frequent words in the song lyrics, excluding nltk stop words, in a strict JSON schema with a single key 'most_frequent_words' which is a list of strings."

attempt_freq_words = 0
max_attempts_freq_words = 3
delay_freq_words = 1

while attempt_freq_words < max_attempts_freq_words:
    try:
        response3 = gemini_model.generate_content(prompt3)

        if not response3.text:
            if response3.prompt_feedback and response3.prompt_feedback.safety_ratings:
                safety_reasons = [f"{s.category}: {s.probability}" for s in response3.prompt_feedback.safety_ratings]
                raise ValueError(f"Model response blocked due to safety concerns for frequent words: {'; '.join(safety_reasons)}")
            else:
                raise ValueError("Model response for frequent words was empty or blocked for an unknown reason.")

        print(response3.text)

        # Extract JSON string for frequent words, handling markdown
        json_string_response3 = response3.text.strip()
        if json_string_response3.startswith("```json") and json_string_response3.endswith("```"):
            json_string_response3 = json_string_response3[7:-3].strip()

        frequent_words_data = json.loads(json_string_response3)
        # Add the frequent words to the song_analysis dictionary
        if 'song_analysis' in locals():
            song_analysis.update(frequent_words_data)
            print("\nMost frequent words successfully added to song_analysis.")
            print(song_analysis)
        else:
            print("\nSong analysis dictionary not available to update with frequent words.")
            print(frequent_words_data)
        break # Break loop on success
    except (json.JSONDecodeError, ValueError) as e:
        attempt_freq_words += 1
        print(f"Error for frequent words (attempt {attempt_freq_words}/{max_attempts_freq_words}): {e}")
        if attempt_freq_words < max_attempts_freq_words:
            print(f"Retrying frequent words in {delay_freq_words} seconds...")
            time.sleep(delay_freq_words)
            delay_freq_words *= 2 # Exponential backoff
    except Exception as e:
        attempt_freq_words += 1
        print(f"An unexpected error occurred for frequent words (attempt {attempt_freq_words}/{max_attempts_freq_words}): {e}")
        if attempt_freq_words < max_attempts_freq_words:
            print(f"Retrying frequent words in {delay_freq_words} seconds...")
            time.sleep(delay_freq_words)
            delay_freq_words *= 2 # Exponential backoff

if 'frequent_words_data' not in locals() and attempt_freq_words == max_attempts_freq_words:
    print(f"Failed to retrieve frequent words after {max_attempts_freq_words} attempts.")


```json
{
  "song_title": "LOYALTY.",
  "artist": "Kendrick Lamar (feat. Rihanna)",
  "genre": [
    "Hip Hop",
    "R&B",
    "Trap"
  ],
  "explicit": true,
  "summary": "LOYALTY. is a dynamic track that explores the complexities of allegiance, trust, and commitment in both personal relationships and the music industry. The song juxtaposes a high-energy, hedonistic chorus focused on immediate gratification and excitement (often delivered by Rihanna's voice) with Kendrick Lamar's introspective verses. Kendrick, using his alias 'Kung Fu Kenny,' asserts his dominance and established legacy while continuously challenging the listener and those around him with the central question: 'Tell me who you loyal to?' He scrutinizes whether dedication lies with money, fame, substances, or genuine human bonds like family and self-respect.",
  "sentiments": [
    "Assertive",
    "Demanding",
    "Questioning",
    "Skeptical",
    "Confident",
    "Hedonistic"
  ],
  "topics_themes": [
    "Loyalty

# stage 4 - Contextual Recommendations
##### Generates a final list of suggested media based on derived, deep context rather than simple title search.

In [ ]:
#stage 4

import json

reccomendations = []

prompt5 = f"Using the data from {song_analysis}, generate a list of similar songs based on derived deep context in a clean and easy to read bulleted list"
response5 = gemini_model.generate_content(prompt5)
print(response5.text)


Based on the deep contextual analysis of "LOYALTY." (focusing on the dynamic blend of Assertive Confidence, Skepticism regarding Trust, Hedonistic R&B/Trap production, and the theme of Fame vs. Authenticity), here is a list of similar songs.

---

### Similar Songs (Deep Context Match)

*   **Travis Scott ft. Drake – "SICKO MODE"**
    *   **Context Match:** Shares the high-energy, ambitious, and multi-phased production style common in modern Trap. It also features a dynamic, high-profile collaboration focused on asserting status, dominance, and addressing internal industry skepticism.
*   **J. Cole – "KOD" (Kids On Drugs)**
    *   **Context Match:** Directly aligns with the themes of **Hedonism and Escapism (Drugs/Sex)** and **Authenticity vs. Materialism**. Like Kendrick, Cole uses a highly focused conceptual track to deliver introspective, critical verses over a modern beat, questioning the true cost of chasing highs.
*   **Drake – "Nonstop"**
    *   **Context Match:** Matches the

#Extra Credit 1 - Curated Playlist
##### Using the analyzed data to generate a new list of real external data points (other songs/artists) with analytical justification

In [ ]:
#extra credit

import json

play_list = []

prompt4 = f"Using the data from {song_analysis}, generate a playlist of 10 similar songs. For each song, provide the 'song_title', 'artist', and a 'justification' (one sentence analytical justification) in a strict JSON array format. Each element of the array should be a JSON object with these three keys."
response4 = gemini_model.generate_content(prompt4)
print(response4.text)

# Extract the playlist string, handling cases where it's wrapped in markdown
playlist_string = response4.text.strip()
if playlist_string.startswith("```json") and playlist_string.endswith("```"):
    playlist_string = playlist_string[7:-3].strip()
elif playlist_string.startswith("```python") and playlist_string.endswith("```"):
    playlist_string = playlist_string[len("```python"): -len("```")].strip()

# Remove the 'playlist =' prefix if it exists
if playlist_string.startswith("playlist ="):
    playlist_string = playlist_string[len("playlist ="):].strip()

try:
    # Parse the string into a list and assign it to play_list
    play_list = json.loads(playlist_string)
    print("\nPlaylist successfully retrieved and saved to 'play_list' variable.")
    print(play_list)
except json.JSONDecodeError as e:
    print(f"JSON decoding error for playlist: {e}")
    print("Raw playlist response:", playlist_string)
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    print("Raw playlist response:", playlist_string)

```json
[
  {
    "song_title": "SICKO MODE",
    "artist": "Travis Scott",
    "justification": "This track shares the complex, genre-bending production, significant beat switches, and themes of hedonism and elevated status present in 'LOYALTY.'"
  },
  {
    "song_title": "DNA.",
    "artist": "Kendrick Lamar",
    "justification": "Similar to 'LOYALTY.,' this song asserts aggressive confidence and status dominance, reinforcing Kendrick's assertive and powerful persona against skeptics."
  },
  {
    "song_title": "The Hills",
    "artist": "The Weeknd",
    "justification": "This R&B/Trap crossover captures the explicit hedonism and commitment issues prevalent in the chorus of 'LOYALTY., focusing on escapism and immediate gratification."
  },
  {
    "song_title": "Headlines",
    "artist": "Drake",
    "justification": "Drake confronts the challenges of fame and the subsequent loyalty tests from those around him, directly addressing the core theme of trust and allegiance."
  },
  {

# Extra Credit 2 - Critical Review
##### Generate a short, 250-word critical review of the song, written in the style of a professional music critic. Requires citing the Sentiments and Themes (from Stage 3) as evidence to support the critical opinion.

In [ ]:
#extra credit 2

prompt6 = f"Using the data from {song_analysis} to support claims and reasoning, generate a 250 word review with a clear markdown heading"
response6 = gemini_model.generate_content(prompt6)
print(response6.text)

## Review: LOYALTY. (Kendrick Lamar feat. Rihanna)

Kendrick Lamar’s "LOYALTY." is a dynamic blend of Hip Hop, R&B, and Trap, built around a central, demanding question: "Tell me who you loyal to?" The track is an introspective examination of allegiance, juxtaposing the complexities of commitment with the temptations of immediate gratification.

The song’s dynamic quality stems from its structural tension. Rihanna’s contribution often anchors the high-energy, hedonistic chorus, emphasizing escapism and excitement—a sentiment supported by the focus on topics like **Hedonism and Escapism** and frequent words like **'high'** and **'geeked'**.

This visceral energy is sharply contrasted by Kendrick Lamar’s verses. Using his alias **'Kung Fu Kenny,'** he takes an **Assertive** and **Skeptical** stance, asserting his dominance and established **Legacy** while simultaneously challenging the authenticity of those around him. Kendrick forces the listener to weigh their dedication between materi

# Extra Credit 3 - New Verse Generation
###### Generate one new, original verse for the song that maintains the existing rhyme scheme, meter, sentiment, and theme. Forcing the model to adhere to structural constraints (meter, rhyme) while maintaining thematic continuity

In [ ]:
#extra credit 3

prompt7 = f"Using the data from {song_analysis} create one new, original verse for the song that maintains the existing rhyme scheme, meter, sentiment, and theme. Maintain continuity"
response7 = gemini_model.generate_content(prompt7)
print(response7.text)

Look at the ledger, tell me what you owe me now
When the confetti drops, everybody trying to hold me down.
I’m the sensei of the system, I don't need a cheat code,
If you ain't moving with me, then you gotta meet the death toll.
Claiming you solid, while you six feet up, you feelin' *high*,
But would you catch the sentence, would you be willing to *die*?
Your dedication is measured, it ain't bought with a dollar sign,
Show me the proof before I sign on the dotted line.
